# Mandelbrot explorer

An example of computation with xeus-quickjs: the Mandelbrot set is computed in JavaScript, and encoded as a
PNG image displayed in an image widget, updated by sliders.

In [ ]:
const CRC_TABLE = Array.from({ length: 256 }, (_, n) => {
  let c = n;
  for (let k = 0; k < 8; k++) {
    c = c & 1 ? 0xedb88320 ^ (c >>> 1) : c >>> 1;
  }
  return c >>> 0;
});

function crc32(bytes) {
  let crc = 0xffffffff;
  for (const byte of bytes) {
    crc = CRC_TABLE[(crc ^ byte) & 0xff] ^ (crc >>> 8);
  }
  return (crc ^ 0xffffffff) >>> 0;
}

/** Encodes RGB pixels in a PNG image, without compression. */
function encodePng(width, height, rgb) {
  const stride = width * 3 + 1;
  const raw = new Uint8Array(stride * height);
  for (let y = 0; y < height; y++) {
    raw.set(rgb.subarray(y * width * 3, (y + 1) * width * 3), y * stride + 1);
  }
  const blocks = Math.ceil(raw.length / 65535);
  const zlib = new Uint8Array(2 + raw.length + blocks * 5 + 4);
  zlib.set([0x78, 0x01]);
  let p = 2;
  for (let i = 0; i < raw.length; i += 65535) {
    const length = Math.min(65535, raw.length - i);
    zlib.set([i + length >= raw.length ? 1 : 0, length & 0xff, length >> 8, ~length & 0xff, (~length >> 8) & 0xff], p);
    zlib.set(raw.subarray(i, i + length), p + 5);
    p += 5 + length;
  }
  let a = 1, b = 0;
  for (const byte of raw) {
    a = (a + byte) % 65521;
    b = (b + a) % 65521;
  }
  new DataView(zlib.buffer).setUint32(p, ((b << 16) | a) >>> 0);

  const chunk = (type, data) => {
    const bytes = new Uint8Array(12 + data.length);
    const view = new DataView(bytes.buffer);
    view.setUint32(0, data.length);
    bytes.set([...type].map(c => c.charCodeAt(0)), 4);
    bytes.set(data, 8);
    view.setUint32(8 + data.length, crc32(bytes.subarray(4, 8 + data.length)));
    return bytes;
  };
  const header = new Uint8Array(13);
  new DataView(header.buffer).setUint32(0, width);
  new DataView(header.buffer).setUint32(4, height);
  header.set([8, 2, 0, 0, 0], 8);
  const parts = [new Uint8Array([137, 80, 78, 71, 13, 10, 26, 10]), chunk('IHDR', header), chunk('IDAT', zlib), chunk('IEND', new Uint8Array(0))];
  const png = new Uint8Array(parts.reduce((sum, part) => sum + part.length, 0));
  parts.reduce((offset, part) => (png.set(part, offset), offset + part.length), 0);
  return png;
}

In [ ]:
/** Computes the Mandelbrot set around a point, and returns its pixels. */
function mandelbrot({ width = 360, height = 240, cx = -0.6, cy = 0, zoom = 1, iterations = 80 } = {}) {
  const pixels = new Uint8Array(width * height * 3);
  const scale = 3 / (width * zoom);
  for (let py = 0; py < height; py++) {
    const y0 = cy + (py - height / 2) * scale;
    for (let px = 0; px < width; px++) {
      const x0 = cx + (px - width / 2) * scale;
      let x = 0, y = 0, i = 0;
      while (x * x + y * y <= 4 && i < iterations) {
        const xt = x * x - y * y + x0;
        y = 2 * x * y + y0;
        x = xt;
        i++;
      }
      const offset = (py * width + px) * 3;
      if (i < iterations) {
        const t = i / iterations;
        pixels[offset] = 9 * (1 - t) * t * t * t * 255;
        pixels[offset + 1] = 15 * (1 - t) * (1 - t) * t * t * 255;
        pixels[offset + 2] = 8.5 * (1 - t) * (1 - t) * (1 - t) * t * 255;
      }
    }
  }
  return pixels;
}

console.time('mandelbrot');
const pixels = mandelbrot();
console.timeEnd('mandelbrot');
display.png(encodePng(360, 240, pixels));

## Exploring with widgets

Move the sliders to zoom on a part of the set.

In [ ]:
const { FloatSlider, FloatLogSlider, IntSlider, Image, VBox, HBox } = Jupyter.widgets;

const image = new Image({ format: 'png' });
const cx = new FloatSlider({ value: -0.6, min: -2, max: 1, step: 0.01, description: 'x', continuous_update: false });
const cy = new FloatSlider({ value: 0, min: -1.2, max: 1.2, step: 0.01, description: 'y', continuous_update: false });
const zoom = new FloatLogSlider({ value: 1, base: 2, min: 0, max: 10, step: 0.1, description: 'zoom', continuous_update: false });
const iterations = new IntSlider({ value: 80, min: 20, max: 300, step: 10, description: 'iterations', continuous_update: false });

const render = () => {
  const options = { cx: cx.value, cy: cy.value, zoom: zoom.value, iterations: iterations.value };
  image.value = encodePng(360, 240, mandelbrot(options));
};
for (const slider of [cx, cy, zoom, iterations]) {
  slider.observe(render, 'value');
}
render();

new HBox({ children: [image, new VBox({ children: [cx, cy, zoom, iterations] })] })